In [ ]:
# ================================================================
# USER CONFIGURATION — set these paths for your environment
# ================================================================
# Root directory for saving anomaly maps — change to your Drive path
# Maps are saved under: {MAPS_SAVE_DIR}/standard/{model_name}/{category}/
MAPS_SAVE_DIR = ''          # e.g. '/content/drive/MyDrive/BachelorsThesis/results/anomaly_maps'
# ================================================================

# Standard Protocol — Real-IAD Benchmark

Evaluates all three DINOv2-based models under the standard multi-view protocol.
All five camera viewpoints are used for both training and evaluation.

Models: AnomalyDINO (memory-based), Dinomaly (reconstruction-based), INP-Former (prototype-based)
Dataset: Real-IAD 512px, 30 categories
Metrics: I-AUROC, S-AUROC, P-AUROC, P-AUPR, AUPRO, Inference Time, Memory Footprint

Note: For a single-category smoke test, see notebooks/01_smoke_test.ipynb

In [7]:
# Reduce CUDA memory fragmentation
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

from google.colab import drive
import sys

drive.mount('/content/drive')

repo_path = '/content/drive/MyDrive/BachelorsThesis'
dataset_root = '/content/drive/MyDrive/datasets/realiad_512'

# Clone repo if not already present, otherwise pull latest
if not os.path.exists(repo_path):
    !git clone https://github.com/PurpleMono/BachelorsThesis.git {repo_path}
    !git -C {repo_path} submodule update --init
else:
    !git -C {repo_path} pull
    !git -C {repo_path} submodule update --init

# Force INP-Former submodule to correct commit with path fixes
!git -C {repo_path}/models/inp_former fetch origin
!git -C {repo_path}/models/inp_former checkout 6041e2b

sys.path.insert(0, repo_path)

# Install dependencies
!pip install anomalib==2.3.3 ADEval einops colorama timm kornia -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

ModuleNotFoundError: No module named 'google'

In [ ]:
# Results paths — consistent across all notebooks
results_path = f'{repo_path}/results'

# Protocol-specific score paths
std_results = f'{results_path}/standard'
cv_results = f'{results_path}/crossview'
abl_results = f'{results_path}/ablation'

# Anomaly map paths
std_maps = f'{MAPS_SAVE_DIR}/standard'
cv_maps = f'{MAPS_SAVE_DIR}/crossview'
maps_abl1_mm = f'{MAPS_SAVE_DIR}/ablation/investigation1/multiclass_multiview'
maps_abl1_sm = f'{MAPS_SAVE_DIR}/ablation/investigation1/singleclass_multiview'
maps_abl1_ms = f'{MAPS_SAVE_DIR}/ablation/investigation1/multiclass_singleview'
maps_abl1_ss = f'{MAPS_SAVE_DIR}/ablation/investigation1/singleclass_singleview'
maps_abl2 = f'{MAPS_SAVE_DIR}/ablation/investigation2'
maps_abl3 = f'{MAPS_SAVE_DIR}/ablation/investigation3'

# Create all directories upfront
for path in [
    std_results,
    cv_results,
    f'{abl_results}/investigation1',
    f'{abl_results}/investigation2',
    f'{abl_results}/investigation3',
    f'{results_path}/weights',
    f'{results_path}/figures',
    std_maps,
    cv_maps,
    maps_abl1_mm,
    maps_abl1_sm,
    maps_abl1_ms,
    maps_abl1_ss,
    maps_abl2,
    maps_abl3,
]:
    os.makedirs(path, exist_ok=True)

print("All results directories ready")

## Dataset Local Copy

Copies Real-IAD from Google Drive to Colab local SSD for faster I/O during training.
Reading directly from Drive is significantly slower than local storage.
This step is skipped if the data is already present on local storage.

In [ ]:
import zipfile, os

zip_dir = '/content/drive/MyDrive/datasets/realiad_512/realiad_512'
target_dir = '/content/realiad_512'
os.makedirs(target_dir, exist_ok=True)

# Also copy JSON files
import shutil
json_src = '/content/drive/MyDrive/datasets/realiad_512/realiad_jsons'
json_dst = '/content/realiad_512/realiad_jsons'
if not os.path.exists(json_dst):
    shutil.copytree(json_src, json_dst)
    print("JSONs copied")

# Unzip each category
for f in sorted(os.listdir(zip_dir)):
    if f.endswith('.zip'):
        category = f.replace('.zip', '')
        if not os.path.exists(f'{target_dir}/{category}'):
            print(f"Unzipping {f}...")
            with zipfile.ZipFile(f'{zip_dir}/{f}', 'r') as z:
                z.extractall(target_dir)
        else:
            print(f"Skipping {category} — already exists")

print("Done")
dataset_root = target_dir
print(f"Active dataset root: {dataset_root}")

In [ ]:
import importlib.util
import pandas as pd
import numpy as np
import gc

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

realiad_utils = load_module("realiad_utils", f"{repo_path}/data/realiad_utils.py")
trainer = load_module("trainer", f"{repo_path}/models/trainer.py")
metrics = load_module("metrics", f"{repo_path}/evaluation/metrics.py")

load_realiad_all = realiad_utils.load_realiad_all

# Dinomaly — now uses official repo via run_inference_dinomaly
train_dinomaly          = trainer.train_dinomaly
run_inference_dinomaly  = trainer.run_inference_dinomaly

# AnomalyDINO — 16-shot multi-class few-shot protocol
train_anomalydino_fewshot = trainer.train_anomalydino_fewshot

# INP-Former
train_inpformer         = trainer.train_inpformer
run_inference_inpformer = trainer.run_inference_inpformer

# Shared inference for AnomalyDINO
run_inference           = trainer.run_inference

# Utility functions
measure_inference_time  = trainer.measure_inference_time
measure_memory_footprint = trainer.measure_memory_footprint

# Metrics
compute_i_auroc   = metrics.compute_i_auroc
compute_s_auroc   = metrics.compute_s_auroc
compute_all_metrics = metrics.compute_all_metrics
REALIAD_CONFIG    = metrics.REALIAD_CONFIG

print("All modules loaded")

## Step 1: Load Real-IAD Dataset

Loads all 30 categories from the official JSON-based split.
Training set contains normal images only.
Test set contains both normal and anomalous images.
Labels follow the JSON anomaly_class convention, consistent with all published papers.

In [ ]:
# Load all 30 Real-IAD categories
df = load_realiad_all(data_root=dataset_root)

train_df = df[(df['split'] == 'train') & (df['label'] == 0)]
test_df = df[df['split'] == 'test']

print(f"Categories: {df['category'].nunique()}")
print(f"Train (normal only): {len(train_df)}")
print(f"Test total: {len(test_df)}")
print(f"Test label distribution:\n{test_df['label'].value_counts()}")

# Save split info for reproducibility
os.makedirs(f'{repo_path}/results', exist_ok=True)
os.makedirs(f'{repo_path}/results/weights', exist_ok=True)
df.to_csv(f'{repo_path}/results/dataset_split_standard.csv', index=False)
print("Dataset split saved")

## Step 2: Dinomaly

Reconstruction-based model with frozen DINOv2-Register ViT-Base/14 encoder.
Trains using the official Dinomaly repository (models/dinomaly submodule).
Loss: global_cosine_hm_percent with progressive hard mining.
50,000 iterations, batch 16, StableAdamW lr=2e-3.

In [ ]:
model_dinomaly = train_dinomaly(
    train_df=train_df,
    n_iterations=50000,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{results_path}/weights/dinomaly_standard.pth'
)

results_dinomaly = run_inference_dinomaly(
    model=model_dinomaly,
    test_df=test_df,
    device='cuda',
    batch_size=16,
    repo_path=repo_path,
    save_anomaly_maps=True,
    maps_save_dir=std_maps
)

results_dinomaly.to_csv(
    f'{std_results}/dinomaly_scores.csv', index=False)
print(f"Dinomaly saved: {len(results_dinomaly)} rows")
print(f"I-AUROC: {compute_i_auroc(results_dinomaly):.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_dinomaly
print("GPU memory cleared")

## Step 3: AnomalyDINO

Training-free nearest-neighbour method with frozen DINOv2-Register ViT-Base/14 encoder.
Patch features from normal training images are stored in a memory bank.
Anomaly scores are computed via nearest-neighbour distance at inference time.
16 reference images per category per viewpoint = 2,400 total training images.
All 30 categories combined into one multi-class memory bank.
Follows the few-shot evaluation protocol from Hofer et al. (2025).

In [ ]:
# Free all GPU memory from Dinomaly and INP-Former before AnomalyDINO
# AnomalyDINO requires maximum available system RAM for CPU memory bank consolidation
# Run this immediately before train_anomalydino
torch.cuda.empty_cache()
gc.collect()
print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

In [ ]:
model_anomalydino = train_anomalydino_fewshot(
    train_df=train_df,
    n_shots=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{results_path}/weights/anomalydino_fewshot.pth'
)

results_anomalydino = run_inference(
    model=model_anomalydino,
    test_df=test_df,
    model_name='AnomalyDINO',
    device='cuda',
    batch_size=1,
    repo_path=repo_path,
    save_anomaly_maps=True,
    maps_save_dir=std_maps
)

results_anomalydino.to_csv(
    f'{std_results}/anomalydino_scores.csv', index=False)
print(f"AnomalyDINO saved: {len(results_anomalydino)} rows")
print(f"I-AUROC: {compute_i_auroc(results_anomalydino):.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_anomalydino
print("GPU memory cleared")

## Step 4: INP-Former

Prototype-based reconstruction model with frozen DINOv2-Register ViT-Base/14 encoder.
Extends Dinomaly with learnable Image-level Normal Prototype tokens that aggregate
global normal patterns at test time to guide feature reconstruction.
Training uses: 100 epochs, batch size 16, StableAdamW lr=1e-3.

In [ ]:
model_inpformer = train_inpformer(
    train_df=train_df,
    dataset_root=dataset_root,
    n_epochs=100,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{repo_path}/results/weights/inpformer_standard.pth'
)

results_inpformer = run_inference_inpformer(
    model=model_inpformer,
    test_df=test_df,
    dataset_root=dataset_root,
    device='cuda',
    batch_size=16,
    repo_path=repo_path,
    save_anomaly_maps=True,
    maps_save_dir=std_maps
)

results_inpformer.to_csv(
    f'{std_results}/inpformer_scores.csv', index=False)
print(f"INP-Former saved: {len(results_inpformer)} rows")
print(f"I-AUROC: {compute_i_auroc(results_inpformer):.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_inpformer
print("GPU memory cleared")

## Step 5: Full Evaluation

Computes all metrics for each model and runs Worst-Group Analysis.
Results are saved to the results/ folder for use in the analysis notebook.

In [ ]:
wga_module = load_module("wga", f"{repo_path}/evaluation/wga.py")
wga_by_category = wga_module.wga_by_category
wga_by_viewpoint = wga_module.wga_by_viewpoint
wga_by_defect_type = wga_module.wga_by_defect_type
find_disagreement_groups = wga_module.find_disagreement_groups
print_wga_summary = wga_module.print_wga_summary

# Load saved scores from new folder structure
results_dinomaly = pd.read_csv(f'{std_results}/dinomaly_scores.csv')
results_anomalydino = pd.read_csv(f'{std_results}/anomalydino_scores.csv')
results_inpformer = pd.read_csv(f'{std_results}/inpformer_scores.csv')

# Summary table
summary = pd.DataFrame({
    'Model': ['Dinomaly', 'AnomalyDINO', 'INP-Former'],
    'I-AUROC': [
        compute_i_auroc(results_dinomaly),
        compute_i_auroc(results_anomalydino),
        compute_i_auroc(results_inpformer)
    ],
    'S-AUROC': [
        compute_s_auroc(results_dinomaly),
        compute_s_auroc(results_anomalydino),
        compute_s_auroc(results_inpformer)
    ],
})

print("=" * 50)
print("STANDARD PROTOCOL RESULTS SUMMARY")
print("=" * 50)
print(summary.round(4).to_string(index=False))

summary.to_csv(f'{std_results}/summary.csv', index=False)
print(f"\nSummary saved to standard/summary.csv")

# Worst-Group Analysis
df_dict = {
    'Dinomaly': results_dinomaly,
    'AnomalyDINO': results_anomalydino,
    'INP-Former': results_inpformer
}
print_wga_summary(df_dict)